
# ENIAC 2026 - Experimentos

Pipeline:

PDFs → Chunking → Embeddings → ChromaDB → Similarity Retrieval → Phi-3 → Avaliação



In [ ]:

!pip install -q langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers pypdf ollama pandas matplotlib pymupdf

# 0. Importação das bibliotecas necessárias

In [48]:
from pathlib import Path
from typing import List, Dict, Any
import hashlib
import copy
import subprocess
import time
import ast

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

import fitz  # PyMuPDF

from langchain_core.documents import Document

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb

from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough

## 1. Construção da Base Vetorial

## Carregamento dos documentos

In [ ]:
DATA_PATH = Path("./sources")

pdfs = list(DATA_PATH.glob("*.pdf"))

print(f"{len(pdfs)} PDFs encontrados")

documents = []

OUTPUT_PATH = Path("./extracted_text")
OUTPUT_PATH.mkdir(exist_ok=True)

for pdf in pdfs:

    doc = fitz.open(pdf)

    for page_num, page in enumerate(doc, start=1):

        blocks = page.get_text("blocks")

        paragraphs = []

        for block in blocks:
            text = block[4].strip()

            if text:
                paragraphs.append(text)

        page_text = "\n\n".join(paragraphs)

        documents.append(
            Document(
                page_content=page_text,
                metadata={
                    "source": pdf.name,
                    "page": page_num,
                },
            )
        )

        # opcional: salvar texto extraído
        output_file = OUTPUT_PATH / f"{pdf.stem}_page_{page_num}.txt"
        output_file.write_text(page_text, encoding="utf-8")

    doc.close()

print(f"Total de páginas: {len(documents)}")

# Limpeza de texto:

- Remoção de trechos desnecessários (referencias, etc...)
- Melhoria da leitura de tabelas e imagens

In [50]:
def clean_references_pages(pages):

    cleaned_pages = []
    remove_remaining_pages = False

    for page in pages:
        # Se já encontramos References, ignoramos as próximas páginas
        if remove_remaining_pages:
            continue
        
        # Dividimos o conteúdo da página em linhas para facilitar a busca por "References"
        lines = page.page_content.split("\n")

        # Procuramos por "References" (ou variações) nas linhas da página
        reference_index = None

        for idx, line in enumerate(lines):
            clean_line = line.strip().lower()
            if clean_line == "references":
                reference_index = idx
                print(clean_line)
                break

        # Encontrou References nesta página
        if reference_index is not None:

            print(f"Encontrado References em: {page.metadata['source']}")

            # Criamos uma nova página apenas com o conteúdo antes de "References"
            new_page = copy.deepcopy(page)
            new_page.page_content = "\n".join( lines[:reference_index]) # O conteúdo da nova página é tudo antes de "References"

            # Se o conteúdo antes de "References" não for vazio, adicionamos a nova página à lista limpa
            if new_page.page_content.strip():
                cleaned_pages.append(new_page)


            # TODAS AS PRÓXIMAS PÁGINAS SÃO DESCARTADAS
            remove_remaining_pages = True
        else:
            # Se não encontramos "References" nesta página, então a mantemos (se ainda não encontramos "References" em páginas anteriores)
            cleaned_pages.append(copy.deepcopy(page))

    return cleaned_pages

In [ ]:
from collections import defaultdict

# Agrupamos os documentos por fonte (nome do PDF) para processar cada documento separadamente
documents_by_source = defaultdict(list)
for doc in documents:
    documents_by_source[doc.metadata["source"]].append(doc)


documents_cleaned = []

for source, pages in documents_by_source.items():
    print("\nProcessando:", source)
    # Ordenamos as páginas pelo número da página para garantir que estão na ordem correta
    pages = sorted(pages, key=lambda x: x.metadata.get("page", 0))

    # Aplicamos a função de limpeza para remover as páginas de referências
    cleaned = clean_references_pages(pages)

    # Imprimimos o número de páginas antes e depois da limpeza para verificar o resultado
    print(f"Antes: {len(pages)}, Depois: {len(cleaned)}")

    # Adicionamos as páginas limpas à lista final de documentos
    documents_cleaned.extend(cleaned)

## Chunking

# Por parágrafos

In [ ]:
chunks = []

for doc in documents_cleaned:

    paragraphs = doc.page_content.split("\n\n")

    for i, paragraph in enumerate(paragraphs):

        paragraph = paragraph.strip()

        if paragraph:
            chunks.append(
                Document(
                    page_content=paragraph,
                    metadata={
                        **doc.metadata,
                        "paragraph": i + 1
                    }
                )
            )

print("Chunks:", len(chunks))

chunks_per_pdf = defaultdict(int)

for chunk in chunks:
    chunks_per_pdf[chunk.metadata["source"]] += 1

chunks_per_pdf

## Ajustando chunks:

Chunks com menos de 50 palavras serão concatenados ao próximo chunk.

In [ ]:
MIN_WORDS = 50

merged_chunks = []
pending_text = ""
pending_metadata = None

for chunk in chunks:

    text = chunk.page_content.strip()

    if not text:
        continue

    word_count = len(text.split())

    if word_count < MIN_WORDS:

        if pending_text:
            pending_text += "\n\n" + text
        else:
            pending_text = text
            pending_metadata = chunk.metadata

        continue

    if pending_text:

        merged_chunks.append(
            Document(
                page_content=f"{pending_text}\n\n{text}",
                metadata=chunk.metadata
            )
        )

        pending_text = ""
        pending_metadata = None

    else:

        merged_chunks.append(
            Document(
                page_content=text,
                metadata=chunk.metadata
            )
        )

# Caso o PDF termine com um chunk pequeno
if pending_text:

    if merged_chunks:

        merged_chunks[-1] = Document(
            page_content=merged_chunks[-1].page_content + "\n\n" + pending_text,
            metadata=merged_chunks[-1].metadata
        )

    else:

        merged_chunks.append(
            Document(
                page_content=pending_text,
                metadata=pending_metadata or {}
            )
        )

print(f"Antes: {len(chunks)}")
print(f"Depois: {len(merged_chunks)}")

## Embeddings e persistência no ChromaDB

In [ ]:

embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-mpnet-base-v2'
)

ids = []

for chunk in chunks:

    unique_text = (
        chunk.metadata["source"]
        + str(chunk.metadata["page"])
        + chunk.page_content
    )

    ids.append(hashlib.md5(unique_text.encode()).hexdigest()) # Gerar um ID único para cada chunk com base no conteúdo e metadados

CHROMA_DB_PERSISTENCE_PATH = './chroma_db_data'

vectorstore = Chroma.from_documents(
    documents=merged_chunks,
    embedding=embedding_model,
    ids=ids,
    persist_directory=CHROMA_DB_PERSISTENCE_PATH
)

vectorstore.persist()


## 2. Configuração do Retriever e LLM

In [55]:
client = chromadb.PersistentClient(path=CHROMA_DB_PERSISTENCE_PATH)

collection = client.get_collection('langchain')

results = collection.get(include=['metadatas'])

num_sources = len({
    m['source']
    for m in results['metadatas']
    if m and 'source' in m
})


vectorstore = Chroma(
    client=client,
    collection_name='langchain',
    embedding_function=embedding_model
)

# Retreiver usando Similarity Search (vetores mais próximos)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


In [ ]:
# Subindo o servidor Ollama para usar o modelo localmente
subprocess.Popen(["ollama","serve"])

In [ ]:
# Instalando o modelo phi3 localmente usando Ollama
!ollama pull phi3

In [58]:

# Definição do modelo de lingugem natural a ser utilizado
llm = ChatOllama(
    model='phi3:latest', # Modelo local rodando via Ollama
    temperature=0,  # Temperatura 0 para respostas mais determinísticas
    num_predict=200 # Limite de tokens que o modelo pode gerar na resposta
)

In [ ]:

#Definindo os prompts a serem usados

# Prompt restritivo

prompt_restritivo = '''
You are an assistant specialized in medical literature.

Use the provided context as your only source of information.

Rules:

- Use only information explicitly stated in the context.
- Do not use prior knowledge.
- Do not use external information.
- Do not guess.
- Do not invent facts.
- Do not answer with information that is not supported by the context.

If the context fully answers the question, provide a complete answer.

If the context only partially answers the question, provide only the information supported by the context and clearly state which information is not available in the provided documents.

If the context does not contain sufficient information to answer the question, output exactly: "I could not find sufficient evidence in the provided documents."

Context:
{context}

Question:
{question}

Answer:
'''

# Prompt não restritivo
prompt_nao_restritivo = '''
You are an assistant specialized in medical literature.

Use the provided context as your primary source of information when answering the question.

If relevant information is not available in the context, you may use your general medical knowledge to provide a complete answer.

Context:
{context}

Question:
{question}

Answer:
'''

## 3. Perguntas do Experimento

In [60]:
QUESTIONS = [
    {
        "id": "P1",
        "category": "present",
        "question": "In cases of a type IA endoleak, where there is sufficient uncovered aneurysm neck length between the lowest renal artery and the endograft fabric, what is the simplest way to resolve this leak?"
    },
    {
        "id": "P2",
        "category": "present",
        "question": "Why is palpation of the popliteal arteries important when evaluating a patient with a suspected diagnosis of abdominal aortic aneurysm?"
    },
    {
        "id": "PP1",
        "category": "partially_present",
        "question": "When comparing open repair and endovascular repair for abdominal aortic aneurysm treatment, which procedure requires a smaller surgical incision for access?"
    },
    {
        "id": "PP2",
        "category": "partially_present",
        "question": "Approximately how many years elapsed between the introduction of open surgery for abdominal aortic aneurysm repair and the first endovascular aortic repair (EVAR)?"
    },
    {
        "id": "A1",
        "category": "absent",
        "question": "Is there any measure that can halt the increase in diameter (growth) of an abdominal aortic aneurysm?"
    },
    {
        "id": "A2",
        "category": "absent",
        "question": "What imaging findings suggest that the etiology of an abdominal aortic aneurysm is mycotic?"
    },
    {
        "id": "OC1",
        "category": "out_of_context",
        "question": "Which pop female artist has achieved the highest-grossing concert tour in history?"
    },
    {
        "id": "OC2",
        "category": "out_of_context",
        "question": "Which football (soccer) player has scored the most goals in FIFA World Cup history?"
    }

]

## 4. Execução e Captura dos Chunks seguida da avaliaçao de similaridade do cosseno

In [61]:
def run_rag_evaluation(
    prompt: str,
    retriever,
    llm,
    questions: List[Dict[str, Any]],
    output_path: str = "./",
    output_filename: str = "rag_evaluation_results"
) -> pd.DataFrame:

    # Função auxiliar: formata chunks
    def format_docs(docs) -> str:
        return "\n\n".join(doc.page_content for doc in docs)

    # Construção do pipeline RAG
    rag_chain = (
        {
            "context": retriever | format_docs, # O retriever retorna os chunks relevantes, que são formatados em um único texto
            "question": RunnablePassthrough()   # A pergunta é passada diretamente para o prompt sem modificação
        }
        | ChatPromptTemplate.from_template(prompt)  # O prompt é aplicado, combinando o contexto e a pergunta
        | llm                                       # O modelo de linguagem gera a resposta com base no prompt
        | StrOutputParser()                         # A resposta é convertida para string simples
    )

    resultados: List[Dict[str, Any]] = []

    # Loop de avaliação
    for q in questions:

        print(f"-> Processando questão {q['id']}...")
        print(f"   Pergunta: {q['question']}")
        print(f"   Categoria: {q['category']}")

        start = time.perf_counter()

        # Recuperação explícita (para logging e análise de chunks)
        docs = retriever.invoke(q["question"])

        # Geração da resposta via RAG
        resposta = rag_chain.invoke(q["question"])

        elapsed = time.perf_counter() - start

        # Extração de fontes únicas
        sources = sorted(
            {
                d.metadata.get("source", "")
                for d in docs
            }
        )

        print(f"   Tempo: {elapsed:.2f}s")

        # Texto completo dos chunks
        chunks_text = "\n\n---\n\n".join(d.page_content for d in docs)

        resultados.append({
            "id": q["id"],
            "category": q["category"],
            "question": q["question"],
            "total_chunks": len(docs),
            "sources": "; ".join(sources),
            "answer": resposta,
            "chunks": chunks_text,
            "chunks_list": [d.page_content for d in docs],
            "elapsed_time": round(elapsed, 2)
        })

    # DataFrame final
    df = pd.DataFrame(resultados)
    df["Manual Evaluation"] = "" # Campo vazio para avaliação manual posterior

    # Salva
    if output_path:
        path = Path(output_path)
        path.mkdir(parents=True, exist_ok=True)
        file_path = path / f"{output_filename}.csv"
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        print(f"Arquivo salvo: {file_path}")

    return df

In [62]:
def _calcular_scores_cosseno(row: pd.Series, embedding_model) -> pd.Series:
    """
    Calcula similaridade cosseno entre resposta e chunks.
    """

    resposta = row["answer"]
    chunks = row["chunks_list"]

    # embedding da resposta (1x)
    resposta_emb = embedding_model.embed_query(resposta)

    scores: List[float] = []

    for chunk in chunks:
        chunk_emb = embedding_model.embed_query(chunk)

        score = cosine_similarity(
            [resposta_emb],
            [chunk_emb]
        )[0][0]

        scores.append(float(score))

    return pd.Series({
        "cosine_scores": scores,
        "cosine_max": max(scores) if scores else None,
        "cosine_mean": float(np.mean(scores)) if scores else None,
        "cosine_min": min(scores) if scores else None
    })


def cosine_similarity_metrics(
    rag_result_df: pd.DataFrame,
    embedding_model,
    output_path: str = "./",
    output_filename: str = "cosine_results"
) -> pd.DataFrame:
    
    print(f"Calculando similaridade cosseno para {output_filename}.csv...")
    df = rag_result_df.copy()

    # Garante que chunks_list seja lista real
    if isinstance(df["chunks_list"].iloc[0], str):
        df["chunks_list"] = df["chunks_list"].apply(eval)

    # Aplica cálculo
    df[
        ["cosine_scores", "cosine_max", "cosine_mean", "cosine_min"]
    ] = df.apply(lambda row: _calcular_scores_cosseno(row, embedding_model), axis=1)

    # Salva
    if output_path:
        path = Path(output_path)
        path.mkdir(parents=True, exist_ok=True)
        file_path = path / f"{output_filename}.csv"
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        print(f"Arquivo salvo: {file_path}")

    return df

In [ ]:
RESULT_PATH_RUN = "./results/run_" + time.strftime("%Y%m%d_%H%M%S")"

In [ ]:
resultados_prompt_restritivo = run_rag_evaluation(llm=llm, retriever=retriever, prompt=prompt_restritivo, questions=QUESTIONS, output_path=f"{RESULT_PATH_RUN}/prompt_restritivo", output_filename="resultados_prompt_restritivo")
avaliacao_similaridade_cosseno_prompt_restritivo = cosine_similarity_metrics(resultados_prompt_restritivo, embedding_model, output_path=f"{RESULT_PATH_RUN}/prompt_restritivo", output_filename="avaliacao_similaridade_cosseno_prompt_restritivo")

In [ ]:
resultados_prompt_nao_restritivo = run_rag_evaluation(llm=llm, retriever=retriever, prompt=prompt_nao_restritivo, questions=QUESTIONS, output_path=f"{RESULT_PATH_RUN}/prompt_nao_restritivo", output_filename="resultados_prompt_nao_restritivo")
avaliacao_similaridade_cosseno_prompt_nao_restritivo = cosine_similarity_metrics(resultados_prompt_nao_restritivo, embedding_model, output_path=f"{RESULT_PATH_RUN}/prompt_nao_restritivo", output_filename="avaliacao_similaridade_cosseno_prompt_nao_restritivo")

# Plotando resultados

In [66]:
# Heatmap plot: Quais chunks específicos estão contribuindo para essa similaridade?

def plot_cosine_scores_heatmap(dataframe=None, title=None, output_path=None, output_filename="heatmap.png"):

    max_chunks = max(len(scores) for scores in dataframe["cosine_scores"])

    heatmap_df = pd.DataFrame(
        dataframe["cosine_scores"].tolist(),
        index=dataframe["id"],
        columns=[f"C{i}" for i in range(1, max_chunks + 1)]
    )

    plt.figure(figsize=(max(10, max_chunks * 0.8), 4))

    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu"
    )

    plt.title(
        title if title
        else "Similaridade de cosseno entre pergunta e chunks recuperados"
    )

    plt.xlabel("Chunk recuperado")
    plt.ylabel("Resposta")
    

    # Salva o heatmap se um caminho de saída for fornecido
    if output_path:
        path = Path(output_path)
        path.mkdir(parents=True, exist_ok=True)
        file_path = path / output_filename
        plt.savefig(file_path, dpi=300, bbox_inches='tight')
        print(f"Heatmap salvo: {file_path}")

    plt.show()

In [67]:
#Violin plot: Como está a distribuição geral das similaridades para cada pergunta?

def plot_cosine_scores_distribution(dataframe, title=None, output_path=None, output_filename="cosine_scores_distribution.png"):

    # Explode para uma linha por chunk
    plot_df = (
        dataframe[["id", "cosine_scores"]]
        .explode("cosine_scores")
        .rename(columns={
            "id": "Resposta",
            "cosine_scores": "Similaridade"
        })
    )

    plot_df["Similaridade"] = plot_df["Similaridade"].astype(float)

    plt.figure(figsize=(10, 5))

    # Distribuição
    sns.violinplot(
        data=plot_df,
        x="Resposta",
        y="Similaridade",
        inner=None
    )

    # Pontos individuais
    sns.swarmplot(
        data=plot_df,
        x="Resposta",
        y="Similaridade",
        size=5,
        color="black"
    )

    if title:
        plt.title(title)
    else:
        plt.title("Distribuição das Similaridades por Resposta")

    plt.xlabel("Resposta")
    plt.ylabel("Cosine Similarity")

    plt.tight_layout()
    

    # Salva o gráfico se um caminho de saída for fornecido
    if output_path:
        path = Path(output_path)
        path.mkdir(parents=True, exist_ok=True)
        file_path = path / output_filename
        plt.savefig(file_path, dpi=300, bbox_inches='tight')
        print(f"Distribuição salva: {file_path}")

    plt.show()

In [ ]:
plot_cosine_scores_heatmap(
    dataframe=avaliacao_similaridade_cosseno_prompt_restritivo, 
    title="Similaridade de cosseno - Prompt Restritivo", 
    output_path=f"{RESULT_PATH_RUN}/prompt_restritivo", 
    output_filename="heatmap_prompt_restritivo.png")

plot_cosine_scores_heatmap(
    dataframe=avaliacao_similaridade_cosseno_prompt_nao_restritivo, 
    title="Similaridade de cosseno - Prompt Não Restritivo", 
    output_path=f"{RESULT_PATH_RUN}/prompt_nao_restritivo", 
    output_filename="heatmap_prompt_nao_restritivo.png")

In [ ]:
plot_cosine_scores_distribution(
    dataframe=avaliacao_similaridade_cosseno_prompt_restritivo, 
    title="Distribuição das Similaridades do Cosseno entre Chunks e Resposta - Prompt Restritivo", 
    output_path=f"{RESULT_PATH_RUN}/prompt_restritivo", 
    output_filename="cosine_distribution_prompt_restritivo.png")

plot_cosine_scores_distribution(
    dataframe=avaliacao_similaridade_cosseno_prompt_nao_restritivo,
    title="Distribuição das Similaridades do Cosseno entre Chunks e Resposta - Prompt Não Restritivo", 
    output_path=f"{RESULT_PATH_RUN}/prompt_nao_restritivo", 
    output_filename="cosine_distribution_prompt_nao_restritivo.png")